In [1]:
import os
import sys
import subprocess
from pathlib import Path

# ==============================================================================
# 1. LOAD WHEELS & INSTALL LIBRARIES
# ==============================================================================
WHEEL_DATASET_DIR = None
print("Searching for the wheels directory...")
for p in Path("/kaggle").rglob("ultralytics*.whl"):
    WHEEL_DATASET_DIR = p.parent
    break

if WHEEL_DATASET_DIR is None:
    raise FileNotFoundError(
        "❌ WHEELS DIRECTORY NOT FOUND!\n"
        "Possible reasons:\n"
        "1. You haven't added the wheels dataset (via 'Add Data' on the right panel).\n"
        "2. If you just downloaded the wheels, the /kaggle/working/wheels folder might have been cleared after a session reset."
    )
print(f"✅ Found wheels directory at: {WHEEL_DATASET_DIR}")

# Proceed with offline installation
print("Installing libraries...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "--no-index", 
    f"--find-links={WHEEL_DATASET_DIR}", 
    "ultralytics", "pycocotools", "pandas", "matplotlib", "seaborn", "tqdm", "sahi"
], check=True)
print("✅ Library installation complete!\n")

# ==============================================================================
# 2. IMPORT LIBRARIES 
# ==============================================================================
import io
import json
import contextlib
import cv2
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from ultralytics import YOLO

Searching for the wheels directory...
✅ Found wheels directory at: /kaggle/input/datasets/tranphungdinh/visdrone-yolo11n-wheels/visdrone_yolo_wheels
Installing libraries...
✅ Library installation complete!

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [2]:
# ==============================================================================
# 3. LOCATE VISDRONE DATASET AND CREATE YAML
# ==============================================================================
print("==================================================")
print("LOCATE DATASET & CREATE YAML")
print("==================================================")
print("Searching for the VisDrone dataset directory...")
DATA_ROOT = None
# Scan for the directory containing standard VisDrone structure
for p in Path('/kaggle/input').rglob('*'):
    if p.is_dir() and (p / 'images/train').exists() and (p / 'images/test-dev').exists():
        DATA_ROOT = p
        break

if not DATA_ROOT:
    raise FileNotFoundError("❌ VisDrone dataset (must contain 'images/train' and 'images/test-dev') not found. Did you Add Data?")

print(f"✅ Found VisDrone dataset at: {DATA_ROOT}")

# Automatically generate YOLO YAML configuration file
yaml_content = f"""
path: {DATA_ROOT.as_posix()}
train: images/train
val: images/val
test: images/test-dev

nc: 10
names: ['pedestrian', 'people', 'bicycle', 'car', 'van', 'truck', 'tricycle', 'awning-tricycle', 'bus', 'motor']
"""
YAML_PATH = '/kaggle/working/visdrone_corrected.yaml'
Path(YAML_PATH).write_text(yaml_content)
print(f"✅ YOLO YAML configuration automatically created at: {YAML_PATH}\n")

LOCATE DATASET & CREATE YAML
Searching for the VisDrone dataset directory...
✅ Found VisDrone dataset at: /kaggle/input/datasets/tranphungdinh/visdrone-yolo-format/VisDrone
✅ YOLO YAML configuration automatically created at: /kaggle/working/visdrone_corrected.yaml



In [ ]:
# ==============================================================================
# 4. GENERAL CONFIGURATION, DYNAMIC MODEL DISCOVERY & INFERENCE (WITH SAHI)
# ==============================================================================
import logging
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

logging.getLogger("sahi").setLevel(logging.ERROR)
print("==================================================")
print("RUNNING INFERENCE ON TEST-DEV SPLIT")
print("==================================================")

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"✅ Using device: {device}")

# ---------------------------------------------------------
USE_FINETUNED_MODEL = True  # False = base model
USE_SAHI = True              # True = slice 640x640, False =  1280 
SAHI_SLICE_SIZE = 640
# ---------------------------------------------------------

if USE_FINETUNED_MODEL:
    BEST_FINETUNED_WEIGHT = "/kaggle/input/datasets/tranphungdinh/visdrone-yolo11m-sahi50/visdrone_experiments/finetune_sahi_640/weights/best.pt"
    weight_path = BEST_FINETUNED_WEIGHT
    run_name = "test_run_finetuned"
    
    print(f"🚀 Loading FINETUNED model: {weight_path}")
    if not Path(weight_path).exists():
        raise FileNotFoundError(f"❌ Finetuned weights file not found: {weight_path}")
else:
    SELECTED_MODEL = "yolo11m.pt"  
    run_name = "test_run_base"
    
    BASE_MODEL_PATH = str(Path("/kaggle/working") / SELECTED_MODEL)
    if not Path(BASE_MODEL_PATH).exists():
        model_candidates = list(Path("/kaggle").rglob(SELECTED_MODEL))
        if model_candidates:
            BASE_MODEL_PATH = str(model_candidates[0])
        else:
            raise FileNotFoundError(f"❌ Weights file not found: {SELECTED_MODEL}. Please check your dataset!")
            
    weight_path = BASE_MODEL_PATH
    print(f"🚀 Loading BASE model from: {weight_path}")

# Initialize common directory path for saving results
save_dir_path = Path(f'/kaggle/working/eval_test_dev_{run_name}')
save_dir_path.mkdir(parents=True, exist_ok=True)
pred_json_path = save_dir_path / "predictions.json"

if not USE_SAHI:
    print(f"\n🚀 Running STANDARD YOLO Inference (imgsz=1280)...")
    model = YOLO(weight_path)
    test_results = model.val(
        data=YAML_PATH,
        split='test',        
        imgsz=1280,
        max_det=500,
        save_json=True,      
        plots=True,          
        project=save_dir_path.parent,
        name=save_dir_path.name 
    )
    
    print(f"✅ Standard YOLO predictions saved at: {pred_json_path}")
    
else:
    print(f"\n🚀 Running SAHI Sliced Inference ({SAHI_SLICE_SIZE}x{SAHI_SLICE_SIZE})...")
    print("Muting SAHI warnings. This might take a while depending on dataset size...")
    
    detection_model = AutoDetectionModel.from_pretrained(
        model_type='yolov8',
        model_path=weight_path,
        confidence_threshold=0.001,
        device=device
    )
    
    # Scan test-dev images
    TEST_IMAGES_DIR = DATA_ROOT / "images/test-dev"
    img_paths = list(TEST_IMAGES_DIR.glob("*.jpg"))
    
    sahi_predictions = []
    
    for img_path in tqdm(img_paths, desc="SAHI Inference on test-dev"):
        result = get_sliced_prediction(
            str(img_path),
            detection_model,
            slice_height=SAHI_SLICE_SIZE,
            slice_width=SAHI_SLICE_SIZE,
            overlap_height_ratio=0.2,
            overlap_width_ratio=0.2,
            postprocess_type="NMS",
            postprocess_match_metric="IOU"
        )
        
        for obj in result.object_prediction_list:
            # CAST TO NATIVE PYTHON FLOAT/INT
            x = float(obj.bbox.minx)
            y = float(obj.bbox.miny)
            w = float(obj.bbox.maxx - x)
            h = float(obj.bbox.maxy - y)
            
            sahi_predictions.append({
                "image_id": img_path.stem,      
                "category_id": int(obj.category.id), 
                "bbox": [x, y, w, h],
                "score": float(obj.score.value)      
            })
            
    # Save predictions to JSON
    with open(pred_json_path, 'w') as f:
        json.dump(sahi_predictions, f)
        
    print(f"✅ SAHI predictions for test-dev saved at: {pred_json_path}")

RUNNING INFERENCE ON TEST-DEV SPLIT
✅ Using device: cuda:0
🚀 Loading FINETUNED model: /kaggle/input/datasets/tranphungdinh/visdrone-yolo11m-sahi50/visdrone_experiments/finetune_sahi_640/weights/best.pt

🚀 Running SAHI Sliced Inference (640x640)...
Muting SAHI warnings. This might take a while depending on dataset size...


SAHI Inference on test-dev:   0%|          | 0/1610 [00:00<?, ?it/s]

Performing prediction on 8 slices.
Performing prediction on 6 slices.
Performing prediction on 2 slices.
Performing prediction on 6 slices.
Performing prediction on 8 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 2 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 6 slices.
Performing prediction on 8 slices.
Performing prediction on 8 slices.
Performing prediction on 6 slices.
Performing predictio

In [4]:
# ==============================================================================
# 5. FUNCTION TO CONVERT YOLO TO COCO AND COMPUTE FULL METRICS
# ==============================================================================
import io
import json
import contextlib
from pathlib import Path
from PIL import Image
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

print("\n==================================================")
print("COMPUTING FULL COCO METRICS (OVERALL & PER CLASS)")
print("==================================================")

TEST_IMAGES_DIR = str(DATA_ROOT / "images/test-dev")
TEST_LABELS_DIR = str(DATA_ROOT / "labels/test-dev")
CLASS_NAMES = ['pedestrian', 'people', 'bicycle', 'car', 'van', 'truck', 'tricycle', 'awning-tricycle', 'bus', 'motor']

def get_metric(coco_eval, metric_type, max_dets_idx, iou_idx=None):
    """
    Clean helper to manually extract specific metrics (AP/AR) from pycocotools matrix.
    - max_dets_idx: index of maxDets array (e.g., 3 corresponds to maxDets=500)
    - iou_idx: None for overall (50-95), 0 for IoU=0.50, 5 for IoU=0.75
    """
    if metric_type == 'ap':
        # precision shape: [T(IoU), R(Recall), K(Class), A(Area), M(maxDets)]
        s = coco_eval.eval['precision']
        if iou_idx is not None:
            s = s[iou_idx:iou_idx+1, :, :, 0, max_dets_idx]
        else:
            s = s[:, :, :, 0, max_dets_idx]
    else:
        # recall shape: [T(IoU), K(Class), A(Area), M(maxDets)]
        s = coco_eval.eval['recall'][:, :, 0, max_dets_idx]
        
    s = s[s > -1]
    return float(np.mean(s)) if len(s) else -1.0


def compute_full_coco_metrics(pred_json_path, img_dir, lbl_dir, class_names, run_name):
    print("🔄 Starting conversion from YOLO TXT to COCO JSON format...")
    
    filename_to_int_id = {}
    
    coco_gt = {
        "images": [],
        "annotations": [],
        "categories": [{"id": i + 1, "name": name} for i, name in enumerate(class_names)]
    }
    
    img_paths = list(Path(img_dir).glob("*.jpg"))
    ann_id = 1
    
    for i, img_path in enumerate(tqdm(img_paths, desc="Creating Ground Truth (test-dev)")):
        img_id = i + 1
        filename_stem = img_path.stem
        filename_to_int_id[filename_stem] = img_id
        
        with Image.open(img_path) as img:
            img_w, img_h = img.size
            
        coco_gt["images"].append({
            "id": img_id,
            "file_name": img_path.name,
            "width": img_w,
            "height": img_h
        })
        
        lbl_path = Path(lbl_dir) / f"{filename_stem}.txt"
        if lbl_path.exists():
            with open(lbl_path, "r") as f:
                lines = f.readlines()
                for line in lines:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cls_id = int(parts[0])
                        x_c, y_c, w, h = map(float, parts[1:5])
                        
                        abs_w = w * img_w
                        abs_h = h * img_h
                        x_min = (x_c * img_w) - (abs_w / 2)
                        y_min = (y_c * img_h) - (abs_h / 2)
                        
                        coco_gt["annotations"].append({
                            "id": ann_id,
                            "image_id": img_id,
                            "category_id": cls_id + 1,
                            "bbox": [x_min, y_min, abs_w, abs_h],
                            "area": abs_w * abs_h,
                            "iscrowd": 0
                        })
                        ann_id += 1
                        
    gt_json_path = "/kaggle/working/testdev_coco_gt.json"
    with open(gt_json_path, "w") as f:
        json.dump(coco_gt, f)
        
    with open(pred_json_path, "r") as f:
        yolo_preds = json.load(f)
        
    pred_categories = [p["category_id"] for p in yolo_preds]
    min_cat = min(pred_categories) if pred_categories else 0
    cat_offset = 1 if min_cat == 0 else 0
    
    coco_dt = []
    for p in yolo_preds:
        orig_id = str(p["image_id"]).split(".")[0] 
        if orig_id in filename_to_int_id:
            new_p = p.copy()
            new_p["image_id"] = filename_to_int_id[orig_id]
            new_p["category_id"] = p["category_id"] + cat_offset
            coco_dt.append(new_p)
            
    if len(coco_dt) == 0:
        raise ValueError("❌ Error: No matching images found between Predictions and Ground Truth!")
            
    dt_json_path = "/kaggle/working/testdev_coco_dt.json"
    with open(dt_json_path, "w") as f:
        json.dump(coco_dt, f)
        
    print("\n🚀 Running PyCOCOTools for comprehensive evaluation...")
    cocoGt = COCO(gt_json_path)
    cocoDt = cocoGt.loadRes(dt_json_path)
    
    # ---------------------------------------------------------
    # A. OVERALL COCO METRICS
    # ---------------------------------------------------------
    cocoEval = COCOeval(cocoGt, cocoDt, 'bbox')
    
    # Standard array: stats[0] to stats[11] will use maxDets=100
    # Custom extraction will use maxDets=500 (index 3)
    cocoEval.params.maxDets = [1, 10, 100, 500] 
    
    cocoEval.evaluate()
    cocoEval.accumulate()
    cocoEval.summarize()
    
    stats = cocoEval.stats
    
    overall_metrics = {
        # --- STANDARD COCO METRICS (maxDets=100) ---
        "AP (50-95)": float(stats[0]),
        "AP50": float(stats[1]),
        "AP75": float(stats[2]),
        "AP small": float(stats[3]),
        "AP medium": float(stats[4]),
        "AP large": float(stats[5]),
        "AR@1": float(stats[6]),
        "AR@10": float(stats[7]),
        "AR@100": float(stats[8]),    
        "AR small": float(stats[9]),
        "AR medium": float(stats[10]),
        "AR large": float(stats[11]),
        
        # --- VISDRONE SPECIFIC METRICS (maxDets=500) ---
        "AP@500": get_metric(cocoEval, 'ap', max_dets_idx=3),
        "AP50@500": get_metric(cocoEval, 'ap', max_dets_idx=3, iou_idx=0),
        "AR@500": get_metric(cocoEval, 'ar', max_dets_idx=3)
    }
    
    overall_out_file = f"/kaggle/working/TestDev_COCO_Overall_Metrics_{run_name}.json"
    with open(overall_out_file, 'w') as f:
        json.dump(overall_metrics, f, indent=4)
    print(f"✅ Overall Results saved at: {overall_out_file}")

    # ---------------------------------------------------------
    # B. PER-CLASS COCO METRICS
    # ---------------------------------------------------------
    print("\n🚀 Computing ALL COCO metrics for EACH class individually...")
    per_class_results = []
    
    for cat in cocoGt.loadCats(cocoGt.getCatIds()):
        cat_id = cat['id']
        cat_name = cat['name']
        
        cocoEval_class = COCOeval(cocoGt, cocoDt, 'bbox')
        cocoEval_class.params.catIds = [cat_id] 
        cocoEval_class.params.maxDets = [1, 10, 100, 500]
        
        with contextlib.redirect_stdout(io.StringIO()):
            cocoEval_class.evaluate()
            cocoEval_class.accumulate()
            cocoEval_class.summarize()
            
        c_stats = cocoEval_class.stats
        per_class_results.append({
            "Class_ID": cat_id,
            "Class_Name": cat_name,
            # Standard stats mapped directly
            "AP (50-95)": float(c_stats[0]),
            "AP50": float(c_stats[1]),
            "AP75": float(c_stats[2]),
            "AP small": float(c_stats[3]),
            "AP medium": float(c_stats[4]),
            "AP large": float(c_stats[5]),
            "AR@1": float(c_stats[6]),
            "AR@10": float(c_stats[7]),
            "AR@100": float(c_stats[8]),
            "AR small": float(c_stats[9]),
            "AR medium": float(c_stats[10]),
            "AR large": float(c_stats[11]),
            # VisDrone maxDets=500 specific stats appended
            "AP@500": get_metric(cocoEval_class, 'ap', max_dets_idx=3),
            "AP50@500": get_metric(cocoEval_class, 'ap', max_dets_idx=3, iou_idx=0),
            "AR@500": get_metric(cocoEval_class, 'ar', max_dets_idx=3)
        })

    per_class_df = pd.DataFrame(per_class_results)
    per_class_out_file = f"/kaggle/working/TestDev_COCO_PerClass_Metrics_{run_name}.csv"
    per_class_df.to_csv(per_class_out_file, index=False)
    
    print(f"✅ Detailed Per-Class Results saved at: {per_class_out_file}")
    
    print("\n--- PER-CLASS PREVIEW (AP & AP50 at maxDets=100) ---")
    print(per_class_df[['Class_Name', 'AP (50-95)', 'AP50', 'AP@500']].to_string(index=False))

# Execute the function
if pred_json_path.exists():
    compute_full_coco_metrics(pred_json_path, TEST_IMAGES_DIR, TEST_LABELS_DIR, CLASS_NAMES, run_name)
else:
    print(f"❌ Could not find prediction JSON file at: {pred_json_path}. YOLO evaluation might have failed.")


COMPUTING FULL COCO METRICS (OVERALL & PER CLASS)
🔄 Starting conversion from YOLO TXT to COCO JSON format...


Creating Ground Truth (test-dev):   0%|          | 0/1610 [00:00<?, ?it/s]


🚀 Running PyCOCOTools for comprehensive evaluation...
loading annotations into memory...
Done (t=0.13s)
creating index...
index created!
Loading and preparing results...
DONE (t=1.49s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=44.87s).
Accumulating evaluation results...
DONE (t=1.86s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.264
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.446
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.270
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.157
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.391
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.505
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.106
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.312
 Average 